# Subfase 6 — Modelado clásico final h5 (Notebook autocontenido)

Este notebook implementa TODO el flujo clásico h5 para BG y CF

In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from bvg_core.config import (
  DATA_MASTER_PATH as DATASET_PATH, CLASSICAL_DIR,
  GLOBAL_SEED, TEST_SIZE, FECHA_COL_2, EMPRESA_COL, 
  FEATURE_EXCLUDE, TARGET_COL
)

from bvg_core.utils import safe_company, sha256_file

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC


In [ ]:
np.random.seed(GLOBAL_SEED)

CLASSICAL_DIR.mkdir(parents=True, exist_ok=True)

COMPANIES = {
    'BANCO GUAYAQUIL S.A.': {'kernel': 'linear', 'C': 0.1, 'probability': True, 'class_weight': 'balanced'},
    'CORPORACION FAVORITA C.A.': {'kernel': 'rbf', 'C': 10.0, 'gamma': 0.01, 'probability': True, 'class_weight': 'balanced'},
}

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset no encontrado: {DATASET_PATH}')

df = pd.read_csv(DATASET_PATH)

required = {FECHA_COL_2, EMPRESA_COL, TARGET_COL}
missing = sorted(required.difference(df.columns))
if missing:
    raise ValueError(f'Columnas faltantes: {missing}')

feature_cols = [c for c in df.columns if c not in FEATURE_EXCLUDE]
if not feature_cols:
    raise ValueError('No se pudieron inferir feature_cols.')

df = df.assign(**{FECHA_COL_2: pd.to_datetime(df[FECHA_COL_2], errors='coerce')})
before = len(df)
df = df.dropna(subset=feature_cols + [TARGET_COL, FECHA_COL_2, EMPRESA_COL]).copy()
after = len(df)
if df.empty:
    raise ValueError('Dataset vacío tras limpieza de NaN.')

nulls = df[feature_cols + [TARGET_COL]].isnull().sum()
if (nulls > 0).any():
    raise ValueError(f'Persisten NaN: {nulls[nulls>0].to_dict()}')

df = df.sort_values([EMPRESA_COL, FECHA_COL_2], kind='stable').reset_index(drop=True)
quality = {
    'total_rows': int(before),
    'after_nan_drop_rows': int(after),
    'dropped_nan_rows': int(before - after),
}
quality

{'total_rows': 2841, 'after_nan_drop_rows': 2831, 'dropped_nan_rows': 10}

In [ ]:
def train_temporal_company(df_in: pd.DataFrame, company: str, params: dict, test_size: int = TEST_SIZE) -> dict:
    d = df_in.loc[df_in[EMPRESA_COL] == company].sort_values(FECHA_COL_2, kind='stable').reset_index(drop=True).copy()
    if len(d) <= test_size:
        raise ValueError(f'Datos insuficientes para {company}: n={len(d)} test_size={test_size}')

    cutoff = len(d) - test_size
    tr = d.iloc[:cutoff].copy()
    te = d.iloc[cutoff:].copy()

    X_train = tr.loc[:, feature_cols].copy()
    y_train = tr.loc[:, TARGET_COL].astype(int).copy()
    X_test = te.loc[:, feature_cols].copy()
    y_test = te.loc[:, TARGET_COL].astype(int).copy()

    svc_kwargs = {
        'kernel': params['kernel'],
        'C': params['C'],
        'class_weight': params['class_weight'],
        'probability': params['probability'],
        'random_state': GLOBAL_SEED,
    }
    if 'gamma' in params:
        svc_kwargs['gamma'] = params['gamma']

    pipe = Pipeline(steps=[
        ('scaler', RobustScaler()),
        ('svc', SVC(
            **svc_kwargs,
        )),
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    metrics = {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'f1': float(f1_score(y_test, y_pred, zero_division=0)),
        'positive_rate_pred': float(np.mean(y_pred)),
        'positive_rate_test': float(np.mean(y_test)),
    }

    meta = {
        'company': company,
        'horizonte': 'h5',
        'random_seed': GLOBAL_SEED,
        'train_start_date': str(tr[FECHA_COL_2].min().date()),
        'train_end_date': str(tr[FECHA_COL_2].max().date()),
        'test_start_date': str(te[FECHA_COL_2].min().date()),
        'test_end_date': str(te[FECHA_COL_2].max().date()),
        'feature_columns': list(feature_cols),
        'target_column': TARGET_COL,
        'params_fixed': params,
        'n_rows_company': int(len(d)),
        'n_train': int(len(X_train)),
        'n_test': int(len(X_test)),
    }

    return {
        'pipeline': pipe,
        'metrics': metrics,
        'meta': meta,
        'X_test': X_test,
    }

def export_and_validate_classical(result: dict) -> dict:
    meta = result['meta']
    company_tag = safe_company(meta['company'])
    prefix = f"{company_tag}_h5"

    model_path = CLASSICAL_DIR / f"{prefix}_pipeline.joblib"
    manifest_path = CLASSICAL_DIR / f"{prefix}_manifest.json"

    joblib.dump(result['pipeline'], model_path)

    manifest = {
        'project': 'tesis_main',
        'subphase': 'subfase-6-modelado-clasico-h5',
        'company': meta['company'],
        'horizonte': 'h5',
        'model_family': 'classical',
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'random_seed': GLOBAL_SEED,
        'train_start_date': meta['train_start_date'],
        'train_end_date': meta['train_end_date'],
        'feature_columns': meta['feature_columns'],
        'target_column': meta['target_column'],
        'sklearn_version': sklearn.__version__,
        'params_fixed': meta['params_fixed'],
        'artifact_checksums_sha256': {model_path.name: sha256_file(model_path)},
        'data_snapshot_hash': hashlib.sha256(json.dumps({
            'company': meta['company'],
            'horizonte': meta['horizonte'],
            'train_start_date': meta['train_start_date'],
            'train_end_date': meta['train_end_date'],
            'n_train': meta['n_train'],
            'feature_columns': meta['feature_columns'],
        }, sort_keys=True).encode('utf-8')).hexdigest(),
        'metrics': result['metrics'],
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

    reloaded = joblib.load(model_path)
    smoke = [int(v) for v in reloaded.predict(result['X_test'].iloc[:5].copy())]

    return {
        'artifact_paths': {'model_path': str(model_path), 'manifest_path': str(manifest_path)},
        'checksum_ok': manifest['artifact_checksums_sha256'][model_path.name] == sha256_file(model_path),
        'smoke_predictions': smoke,
    }

In [8]:
run = {
    'seed': GLOBAL_SEED,
    'dataset_path': str(DATASET_PATH),
    'quality': quality,
    'feature_columns': feature_cols,
    'companies': {},
}

for company, params in COMPANIES.items():
    trained = train_temporal_company(df, company, params, test_size=TEST_SIZE)
    exported = export_and_validate_classical(trained)
    run['companies'][company] = {
        'metrics': trained['metrics'],
        'meta': trained['meta'],
        'export': exported,
    }

summary_path = CLASSICAL_DIR / 'h5_subfase6_run_summary.json'
summary_path.write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding='utf-8')
run

{'seed': 42,
 'dataset_path': 'C:\\Users\\leynd\\OneDrive\\Escritorio\\Tesis\\implementaciones\\desarrollov3\\data\\processed\\BVG_features_svc_master.csv',
 'quality': {'total_rows': 2841,
  'after_nan_drop_rows': 2831,
  'dropped_nan_rows': 10},
 'feature_columns': ['close_last',
  'close_vwap',
  'volume_shares_day',
  'turnover_value_day',
  'n_trades_day',
  'ret_lag_1',
  'ret_lag_2',
  'ret_lag_3',
  'mom_3',
  'mom_5',
  'mom_10',
  'vol_5',
  'vol_10',
  'regime_vol_ratio',
  'ma_5',
  'ma_10',
  'ma_gap',
  'price_vs_ma10',
  'rsi_14',
  'turnover_log1p',
  'volume_log1p',
  'avg_trade_size_log1p',
  'amihud_5',
  'days_since_trade'],
 'companies': {'BANCO GUAYAQUIL S.A.': {'metrics': {'accuracy': 0.8333333333333334,
    'f1': 0.9090909090909091,
    'positive_rate_pred': 1.0,
    'positive_rate_test': 0.8333333333333334},
   'meta': {'company': 'BANCO GUAYAQUIL S.A.',
    'horizonte': 'h5',
    'random_seed': 42,
    'train_start_date': '2019-02-13',
    'train_end_date': '2